# Probability & Statistics Foundations — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicai/learn-notebooks/blob/main/notebooks/foundations/1-2/probability-statistics.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicai/learn-notebooks/blob/main/notebooks/foundations/1-2/probability-statistics.ipynb)

> Companion to the **Probability & Statistics Foundations** article. We rebuild every number in
> the article from scratch with **pure numpy** — no API key — and *watch* the ideas happen:
> odds vs probability, the standard error, and what "95% confidence" really promises.

## What you'll build

1. **Odds <-> probability** — convert both ways and prove the round trip.
2. **Mean, variance, SD** — the three summary numbers of a random variable.
3. **Estimate a risk** from one sample, with its **standard error** and **95% CI**.
4. **A Monte-Carlo proof** that ~95% of 95% confidence intervals cover the true value.
5. **The √n law** — watch the standard error shrink as the sample grows.

**No API key needed.** Everything is deterministic (fixed random seed), so your numbers will
match the article exactly.

### Setup (only numpy)

In [1]:
import numpy as np

rng = np.random.default_rng(0)   # fixed seed -> reproducible numbers
Z = 1.959964                     # normal cutoff leaving 2.5% in each tail (the famous "1.96")
print("ready — numpy", np.__version__)
print(f"Z (95% two-sided cutoff) = {Z}")

ready — numpy 2.2.6
Z (95% two-sided cutoff) = 1.959964


## 1. Odds <-> probability — two currencies for one chance

A **risk** (probability) is events out of *everyone*: `p = events/total`. **Odds** is events per
*non-event*: `odds = p/(1-p)`. Convert back with `p = odds/(1+odds)`. Rainy days: 15 out of 100
give a risk of 15% and odds of about 0.18.

In [2]:
def prob_to_odds(p):  return p / (1 - p)
def odds_to_prob(o):  return o / (1 + o)

p = 0.15
odds = prob_to_odds(p)
print(f"risk p = {p}  ->  odds = p/(1-p) = {odds:.4f}   (15 rainy : 85 dry)")
print(f"odds = {odds:.4f}  ->  p = odds/(1+odds) = {odds_to_prob(odds):.4f}   (round trip: back to 0.15)")
print(f"fair coin: p=0.5  ->  odds = {prob_to_odds(0.5):.1f}   ('even odds')")
print(f"common event p=0.8 -> odds = {prob_to_odds(0.8):.1f}   (risk and odds diverge when common)")

risk p = 0.15  ->  odds = p/(1-p) = 0.1765   (15 rainy : 85 dry)
odds = 0.1765  ->  p = odds/(1+odds) = 0.1500   (round trip: back to 0.15)
fair coin: p=0.5  ->  odds = 1.0   ('even odds')
common event p=0.8 -> odds = 4.0   (risk and odds diverge when common)


## 2. Mean, variance, standard deviation

A random variable is summarized by its **mean** (center), **variance** (average squared distance
from the mean), and **standard deviation** = sqrt(variance) — spread back in the original units.
We reproduce the article's tiny dataset `[2, 4, 4, 4, 5, 5, 7, 9]`.

In [3]:
data = np.array([2, 4, 4, 4, 5, 5, 7, 9])
mu  = data.mean()
var = ((data - mu) ** 2).mean()     # population variance (divide by n)
sd  = np.sqrt(var)
print(f"data      = {data.tolist()}")
print(f"mean  mu  = {mu}")
print(f"variance  = {var}   (average of squared deviations)")
print(f"std dev   = {sd}   (sqrt of variance, in original units)")

# a Bernoulli event (1 with prob p, else 0) has mean p and variance p(1-p)
p = 0.15
print(f"\nBernoulli(p={p}): mean = {p}, variance = p(1-p) = {p*(1-p):.4f}")

data      = [2, 4, 4, 4, 5, 5, 7, 9]
mean  mu  = 5.0
variance  = 4.0   (average of squared deviations)
std dev   = 2.0   (sqrt of variance, in original units)

Bernoulli(p=0.15): mean = 0.15, variance = p(1-p) = 0.1275


## 3. Estimate a risk, with its standard error and 95% CI

You watch **n = 100** days and see **15** rainy ones. The estimate is `p_hat = 0.15`. Its wobble is
the **standard error** `SE = sqrt(p_hat(1-p_hat)/n)`, and the 95% interval is `p_hat +/- 1.96*SE`.

In [4]:
def risk_ci(events, n, z=Z):
    p_hat = events / n
    se    = np.sqrt(p_hat * (1 - p_hat) / n)      # SE of a proportion
    return p_hat, se, (p_hat - z*se, p_hat + z*se)

p_hat, se, (lo, hi) = risk_ci(events=15, n=100)
print(f"p_hat  = {p_hat:.3f}")
print(f"SE     = sqrt({p_hat}*{1-p_hat:.2f}/100) = {se:.4f}")
print(f"95% CI = {p_hat} +/- 1.96*{se:.4f} = ({lo:.3f}, {hi:.3f})")
print(f"\nReport:  risk = {p_hat*100:.0f}%  (95% CI {lo*100:.1f}% to {hi*100:.1f}%)")

p_hat  = 0.150
SE     = sqrt(0.15*0.85/100) = 0.0357
95% CI = 0.15 +/- 1.96*0.0357 = (0.080, 0.220)

Report:  risk = 15%  (95% CI 8.0% to 22.0%)


## 4. Monte-Carlo — do ~95% of 95% CIs cover the truth?

This is what "95% confidence" actually means: not that this one interval has a 95% chance of
holding the truth, but that the **procedure** covers the truth about 95% of the time. We *invent*
the true value (p = 0.15), draw 20,000 fresh samples, build a CI each time, and count the hits.

In [5]:
TRUE_P, N, TRIALS = 0.15, 100, 20000
covered = 0
for _ in range(TRIALS):
    events = rng.binomial(N, TRUE_P)              # one fresh sample of 100 days
    _, _, (lo, hi) = risk_ci(events, N)
    if lo <= TRUE_P <= hi:                         # did THIS interval catch the truth?
        covered += 1

pct = 100 * covered / TRIALS
print(f"true value      = {TRUE_P}")
print(f"intervals built = {TRIALS}")
print(f"covered truth   = {covered}")
print(f"coverage        = {pct:.1f}%   (target 95%)")
print("\n-> the PROCEDURE covers the truth ~95% of the time. A touch under 95% because the")
print("   simple normal (Wald) interval is only approximate for a proportion.")

true value      = 0.15
intervals built = 20000
covered truth   = 18649
coverage        = 93.2%   (target 95%)

-> the PROCEDURE covers the truth ~95% of the time. A touch under 95% because the
   simple normal (Wald) interval is only approximate for a proportion.


## 5. The √n law — SE shrinks as the sample grows

`SE = SD/sqrt(n)`, so precision improves with the **square root** of the sample size. Quadruple
the data and the error only halves.

In [6]:
print("SE of the risk estimate (true p = 0.15):")
prev = None
for n in (100, 400, 1600):
    se_n = np.sqrt(TRUE_P * (1 - TRUE_P) / n)
    note = "" if prev is None else f"   (n x4 -> SE x {se_n/prev:.2f})"
    print(f"  n={n:5d}  SE={se_n:.5f}{note}")
    prev = se_n
print("\n-> to HALVE the SE you need 4x the data; to cut it to a tenth, 100x.")

SE of the risk estimate (true p = 0.15):
  n=  100  SE=0.03571
  n=  400  SE=0.01785   (n x4 -> SE x 0.50)
  n= 1600  SE=0.00893   (n x4 -> SE x 0.50)

-> to HALVE the SE you need 4x the data; to cut it to a tenth, 100x.


## Recap

- **Risk = a proportion** (events / everyone); **odds = p/(1-p)**, and **p = odds/(1+odds)** goes back.
- A random variable has a **mean**, a **variance**, and an **SD** = sqrt(variance); a Bernoulli event has mean `p`, variance `p(1-p)`.
- A sample estimate wobbles by the **standard error** `SE = SD/sqrt(n)`, which shrinks as sqrt(n).
- A **95% CI** is `estimate +/- 1.96*SE`; the Monte-Carlo run showed the *procedure* covers the truth ~95% of the time — not a probability about one fixed interval.

← Back to the **Probability & Statistics Foundations** article for the full explanations, diagrams, and interview Q&A.